In [104]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep
from matplotlib.backends.backend_pdf import PdfPages
from coffea.util import load

def cuts(df_):
    base = (
        (df_['n_ak4jets']   >= 5)       &
        (df_['n_b_outZH']   >= 2)       &
        (df_['ZH_pt']      >= 200) &
        #(df_['nPVGood'] >= 35) &
        (df_['ZH_bbvLscore'] > 0.5) &
        #(df_['ZH_bbvLscore'] < 0.8) &
        (df_['ZH_M']        >= 50)      &
        #(df_['MET_pt'] > 20) &
        (df_['ZH_M']        <= 200)
    )
    return base

# ==========================================
# 1. GLOBAL SETTINGS & CONFIGURATION
# ==========================================
hep.style.use(hep.style.CMS)

COFFEA_DIR = "DataVsMC/" # <-- UPDATE THIS PATH
LUMI = 110 # in fb^-1 

# --- Processes ---
bkg_processes = ['VJets', 'QCD', 'tt_B', 'TTBar', 'SingleTop', 'TTX'] 
sig_processes = ['ttZ', 'ttH']
data_process = 'data_obs'
all_processes = bkg_processes + sig_processes + [data_process]

# --- Systematics ---
syst_bases = [
    'AK4PFPuppi_JER', 'AK4PFPuppi_JES_AbsoluteMPFBias', 'AK4PFPuppi_JES_AbsoluteScale',
    # 'AK4PFPuppi_JES_AbsoluteStat', 'AK4PFPuppi_JES_FlavorQCD', 'AK4PFPuppi_JES_Fragmentation',
    # 'AK4PFPuppi_JES_PileUpDataMC', 'AK4PFPuppi_JES_PileUpEnvelope', 'AK4PFPuppi_JES_PileUpMuZero',
    # 'AK4PFPuppi_JES_PileUpPtBB', 'AK4PFPuppi_JES_PileUpPtEC1', 'AK4PFPuppi_JES_PileUpPtEC2',
    # 'AK4PFPuppi_JES_PileUpPtHF', 'AK4PFPuppi_JES_PileUpPtRef', 'AK4PFPuppi_JES_RelativeBal',
    # 'AK4PFPuppi_JES_RelativeFSR', 'AK4PFPuppi_JES_RelativeJEREC1', 'AK4PFPuppi_JES_RelativeJEREC2',
    # 'AK4PFPuppi_JES_RelativeJERHF', 'AK4PFPuppi_JES_RelativePtBB', 'AK4PFPuppi_JES_RelativePtEC1',
    # 'AK4PFPuppi_JES_RelativePtEC2', 'AK4PFPuppi_JES_RelativePtHF', 'AK4PFPuppi_JES_RelativeSample',
    # 'AK4PFPuppi_JES_RelativeStatEC', 'AK4PFPuppi_JES_RelativeStatFSR', 'AK4PFPuppi_JES_RelativeStatHF',
    # 'AK4PFPuppi_JES_SinglePionECAL', 'AK4PFPuppi_JES_SinglePionHCAL', 'AK4PFPuppi_JES_TimePtEta',
    # 'AK8PFPuppi_JER', 'AK8PFPuppi_JES_AbsoluteMPFBias', 'AK8PFPuppi_JES_AbsoluteScale',
    # 'AK8PFPuppi_JES_AbsoluteStat', 'AK8PFPuppi_JES_FlavorQCD', 'AK8PFPuppi_JES_Fragmentation',
    # 'AK8PFPuppi_JES_PileUpDataMC', 'AK8PFPuppi_JES_PileUpEnvelope', 'AK8PFPuppi_JES_PileUpMuZero',
    # 'AK8PFPuppi_JES_PileUpPtBB', 'AK8PFPuppi_JES_PileUpPtEC1', 'AK8PFPuppi_JES_PileUpPtEC2',
    # 'AK8PFPuppi_JES_PileUpPtHF', 'AK8PFPuppi_JES_PileUpPtRef', 'AK8PFPuppi_JES_RelativeBal',
    # 'AK8PFPuppi_JES_RelativeFSR', 'AK8PFPuppi_JES_RelativeJEREC1', 'AK8PFPuppi_JES_RelativeJEREC2',
    # 'AK8PFPuppi_JES_RelativeJERHF', 'AK8PFPuppi_JES_RelativePtBB', 'AK8PFPuppi_JES_RelativePtEC1',
    # 'AK8PFPuppi_JES_RelativePtEC2', 'AK8PFPuppi_JES_RelativePtHF', 'AK8PFPuppi_JES_RelativeSample',
    # 'AK8PFPuppi_JES_RelativeStatEC', 'AK8PFPuppi_JES_RelativeStatFSR', 'AK8PFPuppi_JES_RelativeStatHF',
    # 'AK8PFPuppi_JES_SinglePionECAL', 'AK8PFPuppi_JES_SinglePionHCAL', 'AK8PFPuppi_JES_TimePtEta',
    'ele_scale', 'ele_smear', 'unclust_En', 'muon_scale', 'muon_smear'
]

weight_vars = [
    'genWeight', 'norm_weight',
    'topptWeight', 'topptWeight_Up', 'topptWeight_Down',
    'ele_reco_sf', 'ele_reco_sfup', 'ele_reco_sfdown',
    'ele_id_sf', 'ele_id_sfup', 'ele_id_sfdown',
    'ele_trig_sf', 'ele_trig_sfup', 'ele_trig_sfdown',
    'mu_id_sf', 'mu_id_sfup', 'mu_id_sfdown',
    'mu_iso_sf', 'mu_iso_sfup', 'mu_iso_sfdown',
    'mu_trig_sf', 'mu_trig_sfup', 'mu_trig_sfdown',
    'puWeight', 'puWeight_up', 'puWeight_down',
    'isr_up', 'isr_down', 'fsr_up', 'fsr_down',
    'mu_r_up', 'mu_r_down', 'mu_f_up', 'mu_f_down',
    'mu_rf_up', 'mu_rf_down',
    'btag_sf', 'btag_sfup', 'btag_sfdown'
]

weight_syst_mapping = {
    'topptWeight': ('topptWeight', 'topptWeight_Up', 'topptWeight_Down'),
    'btag_sf': ('btag_sf', 'btag_sfup', 'btag_sfdown'),
    'ele_reco_sf': ('ele_reco_sf', 'ele_reco_sfup', 'ele_reco_sfdown'),
    'ele_id_sf':   ('ele_id_sf', 'ele_id_sfup', 'ele_id_sfdown'),
    'ele_trig_sf':   ('ele_trig_sf', 'ele_trig_sfup', 'ele_trig_sfdown'),
    'mu_id_sf':    ('mu_id_sf', 'mu_id_sfup', 'mu_id_sfdown'),
    'mu_iso_sf':   ('mu_iso_sf', 'mu_iso_sfup', 'mu_iso_sfdown'),
    'mu_trig_sf':  ('mu_trig_sf', 'mu_trig_sfup', 'mu_trig_sfdown'),
    'puWeight':    ('puWeight_up', 'puWeight_up', 'puWeight_down'),
    'isr':         (None, 'isr_up', 'isr_down'), 
    'fsr':         (None, 'fsr_up', 'fsr_down'),
    'mu_r':        (None, 'mu_r_up', 'mu_r_down'),
    'mu_f':        (None, 'mu_f_up', 'mu_f_down'),
    'mu_rf':       (None, 'mu_rf_up', 'mu_rf_down')
}

systematics = ['nominal'] #+ [f"{s}Up" for s in syst_bases] + [f"{s}Down" for s in syst_bases]

validation_vars = [
    'nPV', 'nPVGood', 'MET_pt', 'MET_phi', 'lep_pt', 'ele_pt', 'muon_pt',
    'lep_eta', 'ele_eta', 'muon_eta', 'n_ak4', 'n_bjet', 'n_ak8',
    'jet1_pt', 'jet2_pt', 'bjet1_pt', #'bjet2_pt', 
    'jet1_eta', 'jet2_eta',
    'bjet1_eta', #'bjet2_eta', 
    'jet1_btag', 'jet2_btag', 'bjet1_btag', #'bjet2_btag',
    'fatjet1_pt', 'fatjet1_eta', 'fatjet1_mass', 'n_b_outZH', 'n_ak4jets', 'ZH_bbvLscore'
]

#validation_vars = ['MET_phi', 'lep_pt', 'lep_eta', 'ZH_bbvLscore']

cut_vars = ['ZH_bbvLscore', 'ZH_M', 'ZH_pt', 'n_b_outZH', 'n_ak4jets', 'MET_pt']
vars_to_extract = validation_vars + ['norm_weight'] + weight_vars + cut_vars

binning_dict = {
    'nPV': (50, 0, 100), 'nPVGood': (50, 0, 100),
    'MET_pt': (40, 0, 800), 'MET_phi': (30, -3.14, 3.14),
    'lep_pt': (40, 0, 800), 'ele_pt': (40, 0, 800), 'muon_pt': (40, 0, 800),
    'lep_eta': (30, -2.5, 2.5), 'ele_eta': (30, -2.5, 2.5), 'muon_eta': (30, -2.4, 2.4),
    'n_ak4jets': (15, 0, 15), 'n_bjet': (10, 0, 10), 'n_ak8': (5, 0, 5),
    'jet1_pt': (40, 0, 1000), 'jet2_pt': (40, 0, 800),
    'bjet1_pt': (40, 0, 800), 'bjet2_pt': (40, 0, 600),
    'jet1_eta': (30, -2.5, 2.5), 'jet2_eta': (30, -2.5, 2.5),
    'bjet1_eta': (30, -2.5, 2.5), 'bjet2_eta': (30, -2.5, 2.5),
    'jet1_btag': (20, 0, 1), 'jet2_btag': (20, 0, 1),
    'bjet1_btag': (20, 0, 1), 'bjet2_btag': (20, 0, 1),
    'fatjet1_pt': (40, 200, 1200), 'fatjet1_eta': (30, -2.5, 2.5), 'fatjet1_mass': (40, 0, 400),
    'ZH_bbvLscore': (30, 0, 1)
}
default_binning = (40, 0, 500)

bkg_colors = {'VJets':'#3f90da', 'QCD':'#ffa90e', 'tt_B':'#bd1f01', 'TTBar':'#94a4a2', 'SingleTop':'#e76300', 'TTX':'#b9ac70'}
sig_colors = {'ttZ':'#832db6', 'ttH':'#a96b59'}
mc_colors = {**bkg_colors, **sig_colors}

process_labels = {
    'VJets': 'V+Jets', 'QCD': 'QCD', 'tt_B': r'$t\bar{t}+bb$',
    'TTBar': r'$t\bar{t} + lf, t\bar{t}+cc$', 'SingleTop': 'single top',
    'TTX': r'$t\bar{t}t\bar{t}, t\bar{t}W, t\bar{t}HW, t\bar{t}Hq$',
    'ttZ': r'$t\bar{t}Z$', 'ttH': r'$t\bar{t}H$', 'data_obs': 'Data', 'QCD': 'QCD', 'VV': 'VV'
}

# ==========================================
# 2. DATA EXTRACTION & WEIGHTING FUNCTIONS
# ==========================================
def extract_nominal_genweights(coffea_dir):
    """Extracts true sum_signOf_genweights, enforcing hardcoded values regardless of string formatting."""
    
    # Store these as pure lowercase, base strings
    target_weights = {
        'tttolnu2q': 480547550.0,
        'ttto2l2nu': 466318140.0,
        'ttto4q': 468705400.0,
        'ttbbtolnu2q': 21007254.0,
        'ttbbto2l2nu': 11683236.0,
        'ttbbto4q': 14012432.0
    }
    
    nom_genweights = {}
    nom_files = [f for f in glob.glob(os.path.join(coffea_dir, "*.coffea")) if 'nom' in os.path.basename(f).lower()]
    
    for file_path in nom_files:
        filein = load(file_path)
        gw_dict = filein.get('sum_signOf_genweights', {})
        
        for dataset, weight in gw_dict.items():
            # If the dataset is nested inside another dict
            if isinstance(weight, dict):
                for sub_dataset, sub_weight in weight.items():
                    # Normalize the string: lowercase and remove year suffixes
                    clean_name = sub_dataset.lower().replace('_2024', '').replace('_2023', '').replace('_2022', '')
                    
                    if clean_name in target_weights:
                        nom_genweights[sub_dataset] = target_weights[clean_name]
                    else:
                        nom_genweights[sub_dataset] = sub_weight
            
            # If the dataset is a direct key/value
            else:
                clean_name = dataset.lower().replace('_2024', '').replace('_2023', '').replace('_2022', '')
                
                if clean_name in target_weights:
                    nom_genweights[dataset] = target_weights[clean_name]
                else:
                    nom_genweights[dataset] = weight
                    
    #print("\n--- Final Forced Genweights ---")
    #for k, v in nom_genweights.items():
    #    if 'tt' in k.lower():
    #        print(f"{k}: {v}")
            
    return nom_genweights

def getZhbbWeight(df_, year=None):
    if 'norm_weight' not in df_.columns:
        return pd.Series(1.0, index=df_.index) 

    weight = df_['norm_weight'].copy()
    
    # ==========================================
    # THE LUMINOSITY PATCH
    # Rescale from 2017 default to 2024 target
    # ==========================================
    old_lumi = 41.529
    target_lumi = 109.950
    weight *= (target_lumi / old_lumi)
    # ==========================================

    gen_w = df_.get('genWeight', pd.Series(1.0, index=df_.index)).fillna(1.0)
    weight *= np.sign(gen_w)
    
    # Be sure to uncomment 'btag_sf' now that the normalization is fixed!
    #sfs = ['puWeight']
    sfs = ['ele_reco_sf', 'ele_id_sf', 'mu_id_sf', 'mu_iso_sf', 'mu_trig_sf', 'puWeight', 'topptWeight', 'btag_sf', 'ele_trig_sf']  
    
    for sf in sfs:
        sf_col = df_.get(sf, pd.Series(1.0, index=df_.index)).fillna(1.0)
        weight *= sf_col
        
    return weight

def load_and_cut_data(variation='nominal', coffea_dir=COFFEA_DIR, year=2024, nom_genweights=None):
    tracked_data = {proc: {} for proc in all_processes}
    all_files = glob.glob(os.path.join(coffea_dir, "*.coffea"))
    
    file_mapping = {
        'ele_scale': 'shape_electron_scale_and_smearing',
        'ele_smear': 'shape_electron_scale_and_smearing',
        'muon_scale': 'shape_muons_scale_and_resolution',
        'muon_smear': 'shape_muons_scale_and_resolution',
        'unclust_En': 'shape_met_type1_calibration'
    }
    
    if variation == 'nominal':
        valid_files = [f for f in all_files if 'nom' in os.path.basename(f).lower()]
    else:
        if variation.endswith('Up'):
            base_syst = variation[:-2]
        elif variation.endswith('Down'):
            base_syst = variation[:-4]
        else:
            base_syst = variation
        
        if 'AK4PFPuppi_' in base_syst or 'AK8PFPuppi_' in base_syst:
            search_string = base_syst.replace('AK4PFPuppi_', 'jec_').replace('AK8PFPuppi_', 'jec_')
        else:
            search_string = file_mapping.get(base_syst, base_syst)
            
        valid_files = [f for f in all_files if search_string.lower() in os.path.basename(f).lower()]
        
        if len(valid_files) == 0:
            print(f"  -> [WARNING] Searched for '{search_string}' but found nothing!")
        
    print(f"Extracting '{variation}' from {len(valid_files)} matching files...")

    for file_path in valid_files:
        filename = os.path.basename(file_path)
        filein = load(file_path)
        genweight_dict = filein.get('sum_signOf_genweights', {})
        
        for raw_proc in filein['columns'].keys():
            mapped_proc = None
            if 'TTTo' in raw_proc: mapped_proc = 'TTBar' if 'tt+B' not in raw_proc else None
            elif 'TTbb' in raw_proc: mapped_proc = 'tt_B' if 'tt+B' in raw_proc else None
            elif 'WJets' in raw_proc or 'DYJets' in raw_proc: mapped_proc = 'VJets'
            elif 'QCD' in raw_proc: mapped_proc = 'QCD'
            elif 'ttH' in raw_proc or 'tth' in raw_proc.lower(): mapped_proc = 'ttH'
            elif 'TTZ' in raw_proc or 'ttz' in raw_proc.lower(): mapped_proc = 'ttZ'
            elif 'TTLL' in raw_proc or 'TTNuNu' in raw_proc: mapped_proc = 'ttZ'
            elif 'DATA' in raw_proc or 'data' in raw_proc.lower(): mapped_proc = 'data_obs'
            elif 'SingleTop' in raw_proc: mapped_proc = 'SingleTop'
            elif 'TTX' in raw_proc: mapped_proc = 'TTX'
            elif 'VV' in raw_proc: mapped_proc = 'VV'
            #print("Raw:", raw_proc, "Mapped", mapped_proc)
            
            if mapped_proc not in all_processes: continue
                
            for dataset in filein['columns'][raw_proc].keys():
                #print("Dataset =", dataset)
                # --- BULLETPROOF GENWEIGHT EXTRACTION ---
                # Since we already forced the hardcoded values into the exact dataset keys 
                # in extract_nominal_genweights, we just do a direct lookup.
                if nom_genweights is not None:
                    genweight = nom_genweights.get(dataset, 1.0)
                    #print("Genweight = ", genweight)
                else:
                    print("Using weird genweight")
                    genweight = genweight_dict.get(dataset, 1.0) 
                    if isinstance(genweight, dict):
                        genweight = genweight.get(dataset, 1.0)
                        
                if genweight == 1.0 and mapped_proc != 'data_obs':
                    print(f"  -> [WARNING] Nominal genweight missing for {dataset}! Defaulting to 1.0.")
                
                try:
                    base_dict = filein['columns'][raw_proc][dataset]['btag_mask'][variation]
                    nom_dict = filein['columns'][raw_proc][dataset]['btag_mask']['nominal']
                except KeyError:
                    continue 
                    
                if mapped_proc == 'data_obs':
                    current_vars_to_extract = validation_vars + cut_vars
                    essential_vars = ['n_b_outZH', 'n_ak4jets'] + cut_vars
                else:
                    current_vars_to_extract = vars_to_extract + cut_vars
                    essential_vars = ['n_b_outZH', 'n_ak4jets', 'norm_weight'] + cut_vars
                
                tmp_data = {}
                skip_dataset = False
                
                for var in current_vars_to_extract:
                    dict_key = f'spanet_output_{var}' if var in ['ttzbb', 'tthbb', 'ttbb', 'ttlf', 'ttcc', 'signal'] else f'events_{var}'
                    
                    if dict_key in base_dict:
                        arr = np.array(base_dict[dict_key].value)
                    elif var in essential_vars:
                        if dict_key in nom_dict:
                            arr = np.array(nom_dict[dict_key].value)
                        else:
                            print(f"🚨 FATAL: Essential '{dict_key}' missing in {mapped_proc} ({dataset})!")
                            skip_dataset = True
                            break 
                    else:
                        continue 
                        
                    if var == 'norm_weight' and mapped_proc != 'data_obs':
                        tmp_data[var] = arr / genweight
                    else:
                        tmp_data[var] = arr
                        
                if skip_dataset: continue 
                
                if tmp_data: 
                    df = pd.DataFrame(tmp_data)
                    if dataset not in tracked_data[mapped_proc]:
                        tracked_data[mapped_proc][dataset] = []
                    tracked_data[mapped_proc][dataset].append(df)

    data_dict = {}
    for proc in all_processes:
        dfs_to_concat = []
        for dataset, branches in tracked_data[proc].items():
            dfs_to_concat.extend(branches)
            
        if dfs_to_concat:
            df = pd.concat(dfs_to_concat, ignore_index=True)
            df = df[cuts(df)] 
            df['tot_weight'] = getZhbbWeight(df, year=year) if proc != 'data_obs' else 1.0
            data_dict[proc] = df
        else:
            data_dict[proc] = None
            
    return data_dict

# ==========================================
# 3. PRE-COMPUTE HISTOGRAMS TO SAVE RAM
# ==========================================
print("\n--- Starting Data Extraction & Histogramming ---")

# --- GRAB GLOBAL NOMINAL GENWEIGHTS ---
print("Extracting true nominal genweights...")
global_nom_genweights = extract_nominal_genweights(COFFEA_DIR)

hist_dict = {v: {p: {} for p in all_processes} for v in validation_vars}

for syst in systematics:
    # Pass them into the loader
    current_data = load_and_cut_data(variation=syst, nom_genweights=global_nom_genweights)
    
    # ==================================================
    # DIAGNOSTIC: PRINT RAW UNWEIGHTED YIELDS
    # ==================================================
    if syst == 'nominal':
        print("\n" + "="*50)
        print(" RAW UNWEIGHTED EVENT COUNTS (PASSING CUTS)")
        print("="*50)
        total_mc_raw = 0
        for proc in all_processes:
            df = current_data[proc]
            raw_count = len(df) if df is not None and not df.empty else 0
            print(f" {proc:<15} : {raw_count:,} events")
            if proc != 'data_obs': 
                total_mc_raw += raw_count
        print("-" * 50)
        print(f" {'Total MC':<15} : {total_mc_raw:,} events")
        print("="*50 + "\n")
    # ==================================================
    
    for var in validation_vars:
        n_bins, x_min, x_max = binning_dict.get(var, default_binning)
        bins = np.linspace(x_min, x_max, n_bins + 1)
        
        for proc in all_processes:
            df = current_data[proc]
            if df is None: continue
                
            if df.empty:
                hist_dict[var][proc][syst] = {'counts': np.zeros(n_bins), 'err2': np.zeros(n_bins)}
                continue
                
            if var in df.columns:
                mask = df[var].notna()
                vals = np.clip(df[var][mask], None, bins[-1])
                weights = df['tot_weight'][mask].copy()
                
                if proc == 'tt_B': weights *= 1.318 # * 1.8 # k-factor
                
                # --- 1. Compute Shape/Nominal Histograms ---
                counts, _ = np.histogram(vals, bins=bins, weights=weights)
                err2, _ = np.histogram(vals, bins=bins, weights=weights**2)
                hist_dict[var][proc][syst] = {'counts': counts, 'err2': err2}
                
                # --- 2. Compute Weight Systematics (ONLY on nominal MC data) ---
                if syst == 'nominal' and proc != 'data_obs':
                    for w_syst, (nom_col, up_col, dn_col) in weight_syst_mapping.items():
                        up_w = weights.copy()
                        dn_w = weights.copy()
                        
                        if nom_col:
                            nom_vals = df.get(nom_col, pd.Series(1.0, index=df.index))[mask].fillna(1.0).values
                            safe_nom = np.where(nom_vals == 0, 1.0, nom_vals) 
                            
                            # --- Fallback to nominal values if UP/DOWN columns are missing ---
                            default_up_dn = pd.Series(nom_vals, index=df[mask].index)
                            up_vals = df.get(up_col, default_up_dn)[mask].fillna(1.0).values
                            dn_vals = df.get(dn_col, default_up_dn)[mask].fillna(1.0).values
                            
                            up_w *= (up_vals / safe_nom)
                            dn_w *= (dn_vals / safe_nom)
                        else:
                            up_vals = df.get(up_col, pd.Series(1.0, index=df.index))[mask].fillna(1.0).values
                            dn_vals = df.get(dn_col, pd.Series(1.0, index=df.index))[mask].fillna(1.0).values
                            up_w *= up_vals
                            dn_w *= dn_vals
                            
                        w_counts_up, _ = np.histogram(vals, bins=bins, weights=up_w)
                        w_err2_up, _ = np.histogram(vals, bins=bins, weights=up_w**2)
                        hist_dict[var][proc][f'{w_syst}Up'] = {'counts': w_counts_up, 'err2': w_err2_up}
                        
                        w_counts_dn, _ = np.histogram(vals, bins=bins, weights=dn_w)
                        w_err2_dn, _ = np.histogram(vals, bins=bins, weights=dn_w**2)
                        hist_dict[var][proc][f'{w_syst}Down'] = {'counts': w_counts_dn, 'err2': w_err2_dn}
            else:
                pass
                
    del current_data

# ==========================================
# 4. PLOTTING FUNCTION
# ==========================================
def plot_variables_to_pdf(var_names, hist_dictionary, syst_base_list, output_filename="Data_MC_Plots.pdf"):
    mc_processes = bkg_processes + sig_processes
    
    with PdfPages(output_filename) as pdf:
        for chunk_start in range(0, len(var_names), 9):
            chunk_vars = var_names[chunk_start : chunk_start + 9]
            
            fig = plt.figure(figsize=(24, 24))
            outer_grid = fig.add_gridspec(3, 3, wspace=0.3, hspace=0.3)
            
            for idx, var_name in enumerate(chunk_vars):
                row, col = idx // 3, idx % 3
                inner_grid = outer_grid[row, col].subgridspec(2, 1, height_ratios=[3, 1], hspace=0.00)
                ax = fig.add_subplot(inner_grid[0])
                rax = fig.add_subplot(inner_grid[1], sharex=ax)
                ax.tick_params(labelbottom=False)
                
                n_bins, x_min, x_max = binning_dict.get(var_name, default_binning)
                bins = np.linspace(x_min, x_max, n_bins + 1)
                bin_centers = 0.5 * (bins[1:] + bins[:-1])
                
                mc_hists, mc_labels, mc_colors_list = [], [], []
                total_mc_counts = np.zeros(n_bins)
                total_mc_stat_err2 = np.zeros(n_bins)
                total_mc_syst_err2 = np.zeros(n_bins)
                
                # Stack Nominal
                for proc in mc_processes:
                    nom_data = hist_dictionary[var_name][proc].get('nominal')
                    if not nom_data: continue
                        
                    counts, err2 = nom_data['counts'], nom_data['err2']
                    yield_total, stat_unc = np.sum(counts), np.sqrt(np.sum(err2))
                    
                    mc_hists.append(counts)
                    mc_labels.append(f"{process_labels.get(proc, proc)} ({yield_total:.1f} ± {stat_unc:.1f})")
                    mc_colors_list.append(mc_colors[proc])
                    
                    total_mc_counts += counts
                    total_mc_stat_err2 += err2

                # Calculate Systematics Envelope
                for s_base in syst_base_list:
                    syst_up_diff, syst_dn_diff = np.zeros(n_bins), np.zeros(n_bins)
                    for proc in mc_processes:
                        nom_data = hist_dictionary[var_name][proc].get('nominal')
                        if not nom_data:
                            continue
                        nom = nom_data['counts']
                        up = hist_dictionary[var_name][proc].get(f'{s_base}Up', {'counts': nom})['counts']
                        dn = hist_dictionary[var_name][proc].get(f'{s_base}Down', {'counts': nom})['counts']
                        syst_up_diff += (up - nom)
                        syst_dn_diff += (dn - nom)
                        
                    max_diff = np.max(np.abs(syst_up_diff))
                    nom_yield = np.sum(total_mc_counts)
                    if nom_yield > 0 and max_diff > (nom_yield * 0.1):
                        print(f"🚨 WARNING: '{s_base}' causing massive shift on '{var_name}'! Max bin diff: {max_diff:.1f}")

                    total_mc_syst_err2 += np.maximum(np.abs(syst_up_diff), np.abs(syst_dn_diff))**2

                # Total Uncertainty = Stat ⊕ Syst
                total_mc_err = np.sqrt(total_mc_stat_err2 + total_mc_syst_err2)

                if mc_hists:
                    hep.histplot(mc_hists, bins=bins, ax=ax, stack=True, histtype='fill', 
                                 label=mc_labels, color=mc_colors_list, sort='yield')

                ax.stairs(values=total_mc_counts + total_mc_err, 
                    baseline=np.clip(total_mc_counts - total_mc_err, 0, None),
                    edges=bins, fill=True, hatch='////', edgecolor='red', facecolor='none', 
                    label=r'Stat $\oplus$ Syst Unc.')

                # --- Process Data ---
                # Convert to float so we can use np.nan for blinding
                data_counts = hist_dictionary[var_name][data_process]['nominal']['counts'].astype(float)
                
                # BLINDING: Hide data for ZH_bbvLscore > 0.8
                if var_name == 'ZH_bbvLscore':
                    blind_mask = bin_centers > 0.8
                    data_counts[blind_mask] = np.nan
                
                data_err = np.sqrt(data_counts)
                data_yield = np.nansum(data_counts) # Use nansum to ignore the blinded bins
                
                if data_yield > 0:
                    data_lbl = f"Data ({data_yield:.0f} ± {np.sqrt(data_yield):.1f})"
                    hep.histplot(data_counts, bins=bins, ax=ax, stack=False, histtype='errorbar', 
                                 color='black', label=data_lbl, yerr=data_err)

                # --- Ratio Panel ---
                with np.errstate(divide='ignore', invalid='ignore'):
                    ratio = data_counts / total_mc_counts
                    
                    # Force absolute values to prevent Matplotlib ValueErrors 
                    # caused by negative MC sum-of-weights in the denominator
                    ratio_err = np.abs(data_err / total_mc_counts) 
                    mc_rel_err = np.abs(total_mc_err / total_mc_counts)

                for arr in [ratio, ratio_err, mc_rel_err]:
                    arr[np.isnan(arr) | np.isinf(arr)] = 0

                # Bounding the downward error: 
                # It must be >= 0, and it cannot be larger than the ratio itself (to prevent crossing 0)
                yerr_down = np.clip(ratio_err, 0, np.maximum(ratio, 0))

                rax.stairs(values=1 + mc_rel_err, 
                           baseline=np.clip(1 - mc_rel_err, 0, None),
                           edges=bins, fill=True, hatch='////', edgecolor='red', facecolor='none')
                           
                rax.errorbar(bin_centers, ratio, yerr=[yerr_down, ratio_err], fmt='ko', markersize=3)
                rax.axhline(1, color='black', linestyle='--')
                
                # --- Styling ---
                ax.set_ylabel("Events")
                ax.legend(loc='upper right', ncol=2, fontsize=10) 
                
                max_val = max(np.max(total_mc_counts), np.max(data_counts))
                ax.set_ylim(0.1, max_val * 100 if max_val > 0 else 100)
                ax.set_yscale('log')
                
                rax.set_xlabel(var_name)
                rax.set_ylabel("Data / MC")
                rax.set_ylim(0, 2.5)
                
                hep.cms.label("Preliminary", data=(data_yield > 0), lumi=LUMI, ax=ax, com=13.6, fontsize=12)

            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig) 
            
    print(f"\nFinished generating plots. Saved to {output_filename}")
    
def plot_pileup_variations(hist_dictionary, output_filename="Pileup_Variations.pdf"):
    print(f"\n--- Generating Pileup Overlay Plots ---")
    mc_processes = bkg_processes + sig_processes
    vars_to_plot = ['nPV', 'nPVGood']
    
    with PdfPages(output_filename) as pdf:
        for var_name in vars_to_plot:
            if var_name not in hist_dictionary: 
                continue
            
            n_bins, x_min, x_max = binning_dict.get(var_name, default_binning)
            bins = np.linspace(x_min, x_max, n_bins + 1)
            bin_centers = 0.5 * (bins[1:] + bins[:-1])
            
            nom_counts = np.zeros(n_bins)
            up_counts = np.zeros(n_bins)
            dn_counts = np.zeros(n_bins)
            
            # Sum up all MC processes
            for proc in mc_processes:
                if 'nominal' in hist_dictionary[var_name][proc]:
                    nom_counts += hist_dictionary[var_name][proc]['nominal']['counts']
                if 'puWeightUp' in hist_dictionary[var_name][proc]:
                    up_counts += hist_dictionary[var_name][proc]['puWeightUp']['counts']
                if 'puWeightDown' in hist_dictionary[var_name][proc]:
                    dn_counts += hist_dictionary[var_name][proc]['puWeightDown']['counts']
            
            fig = plt.figure(figsize=(10, 10))
            gs = fig.add_gridspec(2, 1, height_ratios=[3, 1], hspace=0.05)
            ax = fig.add_subplot(gs[0])
            rax = fig.add_subplot(gs[1], sharex=ax)
            ax.tick_params(labelbottom=False)
            
            # --- Main Plot ---
            hep.histplot(nom_counts, bins=bins, ax=ax, label=f"Nominal (Yield: {np.sum(nom_counts):.1f})", color='black', histtype='step', linewidth=2)
            hep.histplot(up_counts, bins=bins, ax=ax, label=f"PU Up (Yield: {np.sum(up_counts):.1f})", color='red', histtype='step', linewidth=2)
            hep.histplot(dn_counts, bins=bins, ax=ax, label=f"PU Down (Yield: {np.sum(dn_counts):.1f})", color='blue', histtype='step', linewidth=2)
            
            ax.set_ylabel("Events", fontsize=16)
            ax.legend(loc='upper right', fontsize=14)
            hep.cms.label("Preliminary", data=False, lumi=LUMI, ax=ax, com=13.6, fontsize=16)
            
            # --- Ratio Plot ---
            with np.errstate(divide='ignore', invalid='ignore'):
                ratio_up = np.where(nom_counts > 0, up_counts / nom_counts, 0)
                ratio_dn = np.where(nom_counts > 0, dn_counts / nom_counts, 0)
            
            rax.plot(bin_centers, ratio_up, 'r.-', label="Up / Nom", markersize=8)
            rax.plot(bin_centers, ratio_dn, 'b.-', label="Down / Nom", markersize=8)
            rax.axhline(1, color='black', linestyle='--')
            
            rax.set_xlabel(var_name, fontsize=16)
            rax.set_ylabel("Var / Nom", fontsize=14)
            rax.set_ylim(0.5, 1.5)
            rax.legend(loc='upper right', fontsize=10, ncol=2)
            
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)
            
    print(f"Saved pileup comparison plots to {output_filename}")

# ==========================================
# Add this call at the very bottom of your script
# ==========================================
#plot_pileup_variations(hist_dict, "Pileup_Variations.pdf")


# ==========================================
# 5. EXECUTE
# ==========================================
all_syst_bases = syst_bases + list(weight_syst_mapping.keys())
plot_variables_to_pdf(validation_vars, hist_dict, all_syst_bases, "Data_MC_Systematics_YesMuTrig.pdf")


--- Starting Data Extraction & Histogramming ---
Extracting true nominal genweights...
Extracting 'nominal' from 6 matching files...

 RAW UNWEIGHTED EVENT COUNTS (PASSING CUTS)
 VJets           : 91 events
 QCD             : 42 events
 tt_B            : 45,276 events
 TTBar           : 30,992 events
 SingleTop       : 3,877 events
 TTX             : 219,400 events
 ttZ             : 11,170 events
 ttH             : 37,091 events
 data_obs        : 8,080 events
--------------------------------------------------
 Total MC        : 347,939 events


Finished generating plots. Saved to Data_MC_Systematics_YesMuTrig.pdf


In [58]:
def diagnose_systematic(hist_dictionary, var_name, s_base, processes):
    print(f"\n{'='*50}")
    print(f"DIAGNOSTIC REPORT: {s_base} on variable '{var_name}'")
    print(f"{'='*50}")
    
    for proc in processes:
        nom_data = hist_dictionary[var_name][proc].get('nominal')
        if not nom_data:
            continue
            
        nom_counts = nom_data['counts']
        up_counts = hist_dictionary[var_name][proc].get(f'{s_base}Up', {'counts': nom_counts})['counts']
        dn_counts = hist_dictionary[var_name][proc].get(f'{s_base}Down', {'counts': nom_counts})['counts']
        
        nom_yield = np.sum(nom_counts)
        up_yield = np.sum(up_counts)
        dn_yield = np.sum(dn_counts)
        
        # Calculate the relative shift
        up_shift = ((up_yield - nom_yield) / nom_yield * 100) if nom_yield > 0 else 0
        dn_shift = ((dn_yield - nom_yield) / nom_yield * 100) if nom_yield > 0 else 0
        
        # Flag anything suspicious (shifts > 10% or exactly -100%)
        flag = ""
        if abs(up_shift) > 10 or abs(dn_shift) > 10:
            flag = " <-- SUSPICIOUS SHIFT"
        if up_yield == 0 or dn_yield == 0:
            flag = " <-- WARNING: VARIATION ZEROED OUT"

        print(f"\nProcess: {proc} {flag}")
        print(f"  Nominal Yield: {nom_yield:.2f}")
        print(f"  Up Yield:      {up_yield:.2f} ({up_shift:+.1f}%)")
        print(f"  Down Yield:    {dn_yield:.2f} ({dn_shift:+.1f}%)")
        
        # Check for single-bin explosions (outlier weights)
        max_diff_up = np.max(np.abs(up_counts - nom_counts))
        if max_diff_up > (nom_yield * 0.5): # If a single bin changes by 50% of the total yield
             print(f"  [!] ALERT: Massive single-bin shift detected in UP variation (Max shift: {max_diff_up:.2f})")
diagnose_systematic(hist_dict, 'MET_phi', 'btag_sf', bkg_processes+sig_processes)


DIAGNOSTIC REPORT: btag_sf on variable 'MET_phi'

Process: VJets 
  Nominal Yield: 477.45
  Up Yield:      477.45 (+0.0%)
  Down Yield:    477.45 (+0.0%)

Process: QCD 
  Nominal Yield: 442.37
  Up Yield:      453.37 (+2.5%)
  Down Yield:    431.61 (-2.4%)

Process: tt_B 
  Nominal Yield: 6999.82
  Up Yield:      7243.83 (+3.5%)
  Down Yield:    6762.64 (-3.4%)

Process: TTBar 
  Nominal Yield: 43645.68
  Up Yield:      44800.60 (+2.6%)
  Down Yield:    42515.28 (-2.6%)

Process: SingleTop 
  Nominal Yield: 2525.59
  Up Yield:      2525.59 (+0.0%)
  Down Yield:    2525.59 (+0.0%)

Process: TTX 
  Nominal Yield: 167.73
  Up Yield:      167.73 (+0.0%)
  Down Yield:    167.73 (+0.0%)

Process: ttZ 
  Nominal Yield: 291.01
  Up Yield:      300.72 (+3.3%)
  Down Yield:    281.58 (-3.2%)

Process: ttH 
  Nominal Yield: 223.83
  Up Yield:      231.89 (+3.6%)
  Down Yield:    216.00 (-3.5%)


In [10]:
import scipy.stats as stats

def plot_sf_profile(data_dict, x_var, sf_var, output_filename):
    """
    Plots the average Scale Factor vs a Kinematic Variable (Profile Plot)
    """
    print(f"Creating profile plot for {sf_var} vs {x_var}...")
    
    # 1. Combine all MC processes into a single DataFrame
    mc_dfs = []
    for proc in bkg_processes + sig_processes:
        if data_dict.get(proc) is not None and not data_dict[proc].empty:
            mc_dfs.append(data_dict[proc])
            
    if not mc_dfs:
        print("No MC data found to plot!")
        return
        
    df_mc = pd.concat(mc_dfs, ignore_index=True)
    
    # 2. Drop NaNs to ensure the arrays align properly
    mask = df_mc[x_var].notna() & df_mc[sf_var].notna()
    x_vals = np.clip(df_mc[x_var][mask].values, None, binning_dict[x_var][2]) # clip to max bin
    sf_vals = df_mc[sf_var][mask].values
    
    # 3. Get binning from your existing dictionary
    n_bins, x_min, x_max = binning_dict.get(x_var, default_binning)
    bins = np.linspace(x_min, x_max, n_bins + 1)
    bin_centers = 0.5 * (bins[:-1] + bins[1:])
    x_err = 0.5 * (bins[1:] - bins[:-1])
    
    # 4. Calculate Mean and Standard Error per bin
    counts, _ = np.histogram(x_vals, bins=bins)
    sum_sf, _ = np.histogram(x_vals, bins=bins, weights=sf_vals)
    sum_sf2, _ = np.histogram(x_vals, bins=bins, weights=sf_vals**2)
    
    # Safely divide to get mean and variance
    mean_sf = np.divide(sum_sf, counts, out=np.zeros_like(sum_sf), where=counts!=0)
    mean_sf2 = np.divide(sum_sf2, counts, out=np.zeros_like(sum_sf2), where=counts!=0)
    
    variance = mean_sf2 - (mean_sf**2)
    # Standard error of the mean = std_dev / sqrt(N)
    std_err = np.sqrt(np.maximum(variance, 0)) / np.sqrt(np.maximum(counts, 1))
    
    # 5. Plotting
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Mask out empty bins for plotting
    valid = counts > 0 
    
    ax.errorbar(bin_centers[valid], mean_sf[valid], 
                xerr=x_err[valid], yerr=std_err[valid], 
                fmt='ko', markersize=5, label='MC Average SF')
    
    # Styling
    ax.axhline(1.0, color='gray', linestyle='--', alpha=0.7)
    ax.set_xlabel(x_var)
    ax.set_ylabel(f"Average {sf_var}")
    
    # Dynamically set y-limits to zoom in on the SF variations
    y_mean = np.mean(mean_sf[valid])
    ax.set_ylim(y_mean * 0.85, y_mean * 1.15) 
    
    hep.cms.label("Preliminary", data=False, lumi=LUMI, ax=ax, com=13.6)
    ax.legend(loc='best')
    
    plt.savefig(output_filename, bbox_inches='tight')
    plt.close(fig)
    print(f" -> Saved {output_filename}")
    
# ==========================================
# 6. PLOT SCALE FACTORS VS KINEMATICS
# ==========================================
# Load just the nominal data (no systematics needed for base profile)
print("\n--- Extracting Nominal Data for SF Profiling ---")
nominal_data = load_and_cut_data(variation='nominal')
print(nominal_data['TTBar'].columns)
# Plot SF vs pT
plot_sf_profile(
    data_dict=nominal_data, 
    x_var='muon_pt', 
    sf_var='mu_trig_sf', 
    output_filename="MuTrigSF_vs_Pt.pdf"
)

# Plot SF vs Eta
plot_sf_profile(
    data_dict=nominal_data, 
    x_var='muon_eta', 
    sf_var='mu_trig_sf', 
    output_filename="MuTrigSF_vs_Eta.pdf"
)


--- Extracting Nominal Data for SF Profiling ---
Extracting 'nominal' from 5 matching files...
Index(['nPV', 'nPVGood', 'MET_pt', 'MET_phi', 'lep_pt', 'ele_pt', 'muon_pt',
       'lep_eta', 'ele_eta', 'muon_eta', 'n_ak4', 'n_bjet', 'n_ak8', 'jet1_pt',
       'jet2_pt', 'bjet1_pt', 'bjet2_pt', 'jet1_eta', 'jet2_eta', 'bjet1_eta',
       'bjet2_eta', 'jet1_btag', 'jet2_btag', 'bjet1_btag', 'bjet2_btag',
       'fatjet1_pt', 'fatjet1_eta', 'fatjet1_mass', 'n_b_outZH', 'n_ak4jets',
       'norm_weight', 'genWeight', 'topptWeight', 'topptWeight_Up',
       'topptWeight_Down', 'ele_reco_sf', 'ele_reco_sfup', 'ele_reco_sfdown',
       'ele_id_sf', 'ele_id_sfup', 'ele_id_sfdown', 'mu_id_sf', 'mu_id_sfup',
       'mu_id_sfdown', 'mu_iso_sf', 'mu_iso_sfup', 'mu_iso_sfdown',
       'mu_trig_sf', 'mu_trig_sfup', 'mu_trig_sfdown', 'puWeight',
       'puWeight_up', 'puWeight_down', 'isr_up', 'isr_down', 'fsr_up',
       'fsr_down', 'mu_r_up', 'mu_r_down', 'mu_f_up', 'mu_f_down', 'mu_rf_up',
       

In [16]:
def plot_sf_eta_pt_map(data_dict, sf_var='mu_trig_sf', output_filename="MuTrigSF_EtaPt_Map.pdf"):
    """
    Plots a 2D Map (Eta vs pT) where the color represents the average Scale Factor.
    """
    print(f"Creating 2D Eta-pT map for {sf_var}...")
    
    # 1. Combine all MC processes into a single DataFrame
    mc_dfs = []
    for proc in bkg_processes + sig_processes:
        if data_dict.get(proc) is not None and not data_dict[proc].empty:
            mc_dfs.append(data_dict[proc])
            
    if not mc_dfs:
        print("No MC data found to plot!")
        return
        
    df_mc = pd.concat(mc_dfs, ignore_index=True)
    
    # --- Data Validations ---
    if sf_var not in df_mc.columns:
        print(f"🚨 WARNING: '{sf_var}' not found in the DataFrame! Skipping 2D plot.")
        return
        
    if 'muon_pt' not in df_mc.columns or 'muon_eta' not in df_mc.columns:
        print(f"🚨 WARNING: Kinematic variables missing! Check validation_vars.")
        return

    # 2. Extract and clean values
    x_var, y_var = 'muon_eta', 'muon_pt'
    mask = df_mc[x_var].notna() & df_mc[y_var].notna() & df_mc[sf_var].notna()
    
    x_vals = df_mc[x_var][mask].values
    y_vals = df_mc[y_var][mask].values
    sf_vals = df_mc[sf_var][mask].values
    
    # 3. Get binning 
    # Example: 12 bins for eta (width of 0.4 per bin)
    nx, xmin, xmax = (12, -2.4, 2.4) 
    
    # Example: 10 bins for pT (width of 50 GeV per bin, capped at 500 to avoid empty high-pT bins)
    ny, ymin, ymax = (16, 0, 800)
    
    x_bins = np.linspace(xmin, xmax, nx + 1)
    y_bins = np.linspace(ymin, ymax, ny + 1)
    
    # 4. Calculate 2D Means using numpy
    counts, x_edges, y_edges = np.histogram2d(x_vals, y_vals, bins=[x_bins, y_bins])
    sum_sf, _, _ = np.histogram2d(x_vals, y_vals, bins=[x_bins, y_bins], weights=sf_vals)
    
    # Divide sum by counts to get the mean, avoiding division by zero
    with np.errstate(divide='ignore', invalid='ignore'):
        mean_sf = np.where(counts > 0, sum_sf / counts, np.nan)
        
    # 5. Plotting
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Transpose mean_sf (.T) because pcolormesh expects (Y, X)
    cmap = plt.get_cmap('viridis').copy()
    cmap.set_bad(color='white') # Set bins with 0 stats to white
    
    im = ax.pcolormesh(x_edges, y_edges, mean_sf.T, cmap=cmap, shading='flat')
    
    # Add Colorbar
    cbar = fig.colorbar(im, ax=ax, pad=0.02)
    cbar.set_label(f"Average {sf_var}", fontsize=18)
    
    # Styling
    ax.set_xlabel(r"Muon $\eta$")
    ax.set_ylabel(r"Muon $p_{T}$ [GeV]")
    
    # Optional: Set a logical max limit for pT if you have a long tail
    # ax.set_ylim(ymin, 500) 
    
    hep.cms.label("Preliminary", data=False, lumi=LUMI, ax=ax, com=13.6)
    
    plt.savefig(output_filename, bbox_inches='tight')
    plt.close(fig)
    print(f" -> Saved {output_filename}")
plot_sf_eta_pt_map(
    data_dict=nominal_data, 
    sf_var='mu_trig_sf', 
    output_filename="MuTrigSF_EtaPt_2D_Map.pdf"
)

Creating 2D Eta-pT map for mu_trig_sf...
 -> Saved MuTrigSF_EtaPt_2D_Map.pdf
